# Architekturen A und B auf Google Colab

Trainiert Architektur A (ResNet-18 mit drei Köpfen: Hauptschild, Zusatzschild vorhanden, Zusatzzeichen) und Architektur B (U-Net + Klassifikator je Zusatzschild) auf Synset Signset Germany und echten Mapillary-Ausschnitten und misst beide auf dem echten Testsatz (Aachen, Eifel, Köln).

**Vorher, auf deinem Rechner:**
1. Den Code committen und pushen (Colab klont das Repo).
2. `python scripts/daten/pack_colab_data.py` ausführen und die entstandene `data/colab_daten.zip` in Google Drive nach `MyDrive/dlacv_gtsrb/` hochladen. Sie enthält die echten Ausschnitte, die wegen `.gitignore` nicht im Repo liegen.

**In Colab:** *Laufzeit → Laufzeittyp ändern → T4 GPU*, dann alle Zellen der Reihe nach ausführen.

Die vorbereiteten Synset-Daten (ca. 4 GB) werden beim ersten Mal in Drive gesichert; spätere Sitzungen kopieren sie nur noch.

## 1. GPU prüfen

In [ ]:
import torch
print('PyTorch', torch.__version__)
assert torch.cuda.is_available(), 'Keine GPU! Laufzeit -> Laufzeittyp ändern -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Code holen und Abhängigkeiten installieren

Bei einem privaten Repo: `https://<user>:<token>@github.com/...` (Token nicht im Notebook committen).

In [ ]:
REPO_URL = 'https://github.com/tuananh-aa-2001/Traffic-Sign-Recognition-System.git'  # anpassen
BRANCH = 'pipeline-fixes-colab'  # anpassen, falls der Code auf einem anderen Branch liegt

import os
%cd /content
if not os.path.exists('/content/project'):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} project
%cd /content/project
!git log --oneline -3
assert os.path.exists('scripts/training/train_multitask.py'), 'scripts/training/train_multitask.py fehlt: lokalen Code committen und pushen'
!pip install -q -r requirements.txt

## 3. Google Drive verbinden und Konfiguration schreiben

In [ ]:
import yaml
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
DRIVE_DIR = '/content/drive/MyDrive/dlacv_gtsrb'
os.makedirs(f'{DRIVE_DIR}/multitask', exist_ok=True)   # auch für das Trainingslog

with open('configs/multitask.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['training']['num_workers'] = os.cpu_count()
cfg['output']['checkpoint'] = f'{DRIVE_DIR}/multitask/multitask_best.pth'   # überlebt einen Sitzungsabbruch
cfg['output']['results_dir'] = f'{DRIVE_DIR}/multitask'
with open('configs/colab_multitask.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
print(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

# Architektur B: Ergebnisse ebenfalls in Drive
with open('configs/zweistufig.yaml', encoding='utf-8') as f:
    cfg_b = yaml.safe_load(f)
cfg_b['training']['num_workers'] = os.cpu_count()
cfg_b['output'] = {'unet': f'{DRIVE_DIR}/zweistufig/b_unet.pth', 'klassifikator': f'{DRIVE_DIR}/zweistufig/b_zusatz.pth',
                   'results_dir': f'{DRIVE_DIR}/zweistufig'}
os.makedirs(f'{DRIVE_DIR}/zweistufig', exist_ok=True)
with open('configs/colab_zweistufig.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg_b, f, allow_unicode=True, sort_keys=False)

## 4. Echte Ausschnitte entpacken

Aus `colab_daten.zip` in Drive (siehe oben, `scripts/daten/pack_colab_data.py`).

In [ ]:
ZIP = f'{DRIVE_DIR}/colab_daten.zip'
assert os.path.exists(ZIP), f'{ZIP} fehlt: lokal python scripts/daten/pack_colab_data.py ausführen und hochladen'
!unzip -q -o "{ZIP}" -d /content/project
!wc -l crops/labels.csv crops/geprueft.csv real_world_mapillary/labels.csv

## 5. Synset vorbereiten (einmalig ca. 20–30 Minuten)

Lädt die Parquet-Dateien der Cycles-Variante (Training 7,4 GB, Validierung 1,8 GB), bringt jedes Bild auf 128×128 und speichert sie als Array. Für Architektur B werden zusätzlich die Klassenkarten (Hauptschild, andere Schilder, Mast) gespeichert. Das Ergebnis wird in Drive gesichert; bei einer neuen Sitzung wird es nur kopiert.

In [ ]:
SYN_DRIVE = f'{DRIVE_DIR}/synset'
URL = 'https://huggingface.co/datasets/FraunhoferIOSB/Synset-Signset-Germany/resolve/main/cycles/{}.parquet'
os.makedirs('data/synset', exist_ok=True)
for split in ['validation', 'train']:
    npy = f'{SYN_DRIVE}/{split}_bilder.npy'
    karten = f'{SYN_DRIVE}/{split}_klassen.npy'
    if all(os.path.exists(f'{SYN_DRIVE}/{split}_{t}') for t in ['bilder.npy', 'klassen.npy', 'labels.csv']):
        print(f'{split}: aus Drive kopieren')
        !cp "{SYN_DRIVE}/{split}_bilder.npy" "{SYN_DRIVE}/{split}_klassen.npy" "{SYN_DRIVE}/{split}_labels.csv" data/synset/
    else:
        print(f'{split}: herunterladen und vorbereiten')
        !wget -q -O /content/{split}.parquet {URL.format(split)}
        !python scripts/daten/prepare_synset.py --split {split} --quelle /content/{split}.parquet --segmentierung
        !rm /content/{split}.parquet
        os.makedirs(SYN_DRIVE, exist_ok=True)
        !cp data/synset/{split}_bilder.npy data/synset/{split}_klassen.npy data/synset/{split}_labels.csv "{SYN_DRIVE}/"
!cp configs/synset_klassen.json "{SYN_DRIVE}/" 2>/dev/null; ls -lh data/synset

## 6. Kurzer Probelauf

Prüft in einer Minute, ob alles zusammenpasst, bevor das lange Training startet.

In [ ]:
!python scripts/training/train_multitask.py --smoke

## 7. Training

15 Epochen à 30.000 Bilder (etwa 30 % echte Ausschnitte je Batch). Das beste Modell wird nach Synset-Validierung und echter Validierung (Bonn, Düsseldorf, Münster, Berlin, München, Hamburg) gewählt, nie nach dem Test. Die Laufzeit für den Bericht notieren.

In [ ]:
!python scripts/training/train_multitask.py --config configs/colab_multitask.yaml 2>&1 | tee "{DRIVE_DIR}/multitask/train_log.txt"

## 8. Ergebnis auf dem echten Testsatz

In [ ]:
import json, pandas as pd
with open(f'{DRIVE_DIR}/multitask/ergebnis.json', encoding='utf-8') as f:
    erg = json.load(f)
for teil in ['echt_test', 'synset_val']:
    r = erg[teil]
    print(teil, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items() if k != 'zusatz_f1_je_klasse'})
je_klasse = erg['echt_test'].get('zusatz_f1_je_klasse', {})
pd.DataFrame(je_klasse).T.sort_values('anzahl', ascending=False)

In [ ]:
import matplotlib.pyplot as plt
with open(f'{DRIVE_DIR}/multitask/verlauf.json', encoding='utf-8') as f:
    verlauf = json.load(f)
ep = [v['epoche'] for v in verlauf]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k in ['haupt', 'da', 'zusatz']:
    ax[0].plot(ep, [v['verlust'][k] for v in verlauf], label=k)
ax[0].set_title('Trainingsverlust je Kopf'); ax[0].set_xlabel('Epoche'); ax[0].legend()
ax[1].plot(ep, [v['synset_val'].get('haupt_acc') for v in verlauf], label='Hauptschild (Synset-Val)')
ax[1].plot(ep, [v['echt_val'].get('zusatz_exakt') for v in verlauf], label='Zusatz exakt (echt-Val)')
ax[1].set_title('Validierung'); ax[1].set_xlabel('Epoche'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/multitask/verlauf.png', dpi=120); plt.show()

## 9. Architektur B: U-Net + Klassifikator je Zusatzschild

Stufe 1: U-Net auf den Synset-Klassenkarten (Hintergrund, Hauptschild, andere Schilder, Mast). Stufe 2: Klassifikator auf einzelnen echten Zusatzschildern (Boxen von Mapillary). Danach die ganze Pipeline auf dem echten Testsatz: Ausschnitt → U-Net → Boxen → Klassifikator → Zusatzzeichen und Zielgruppe.

In [ ]:
!python scripts/training/train_zweistufig.py alles --smoke
!python scripts/training/train_zweistufig.py unet --config configs/colab_zweistufig.yaml 2>&1 | tee "{DRIVE_DIR}/zweistufig/unet_log.txt"
!python scripts/training/train_zweistufig.py klassifikator --config configs/colab_zweistufig.yaml 2>&1 | tee "{DRIVE_DIR}/zweistufig/klassifikator_log.txt"
!python scripts/training/train_zweistufig.py auswerten --config configs/colab_zweistufig.yaml 2>&1 | tee "{DRIVE_DIR}/zweistufig/auswerten_log.txt"

In [ ]:
with open(f'{DRIVE_DIR}/zweistufig/auswerten.json', encoding='utf-8') as f:
    b = json.load(f)['echt_test']
a = erg['echt_test']
pd.DataFrame({'A (End-to-End)': {k: a.get(k) for k in ['da_acc', 'zusatz_exakt', 'zielgruppe_korrekt_relevant']},
              'B (zweistufig)': {k: b.get(k) for k in ['da_acc', 'zusatz_exakt', 'zielgruppe_korrekt_relevant']}})

## 10. Experiment E1: synthetisch gegen echt beim Hauptschild

Gleiche Architektur wie die GTSRB-Baseline (ResNet-18, 48×48), gleiche 43 Klassen. Trainiert wird nur das Synset-Modell; das GTSRB-Modell kommt aus dem GTSRB-Notebook (`MyDrive/dlacv_gtsrb/checkpoints/best_model.pth`). Ergebnis ist eine 2×2-Tabelle: trainiert auf Synset bzw. GTSRB, getestet auf Synset-Validierung bzw. GTSRB-Testset.

In [ ]:
!python src/utils/download_dataset.py --data_dir data/raw
os.makedirs('models/checkpoints/e1_synset', exist_ok=True)
GTSRB_CKPT = f'{DRIVE_DIR}/checkpoints/best_model.pth'
if os.path.exists(GTSRB_CKPT):
    !cp "{GTSRB_CKPT}" models/checkpoints/best_model.pth
else:
    print('GTSRB-Modell nicht in Drive gefunden: Zeile "trainiert auf GTSRB" bleibt leer.')
SYNSET_CKPT = f'{DRIVE_DIR}/e1/synset_model.pth'
if os.path.exists(SYNSET_CKPT):          # schon trainiert: nur auswerten
    !cp "{SYNSET_CKPT}" models/checkpoints/e1_synset/best_model.pth
!python scripts/auswertung/experiment_e1.py 2>&1 | tee e1_log.txt
!mkdir -p "{DRIVE_DIR}/e1"
!cp results/e1/ergebnis.json e1_log.txt "{DRIVE_DIR}/e1/"
!cp results/e1/training_curves.png "{DRIVE_DIR}/e1/" 2>/dev/null || true
!cp -n models/checkpoints/e1_synset/best_model.pth "{DRIVE_DIR}/e1/synset_model.pth"

In [ ]:
with open('results/e1/ergebnis.json', encoding='utf-8') as f:
    e1 = json.load(f)
tab = pd.DataFrame({k: {'Accuracy': v['accuracy'], 'Makro-F1': v['makro_f1'], 'n': v['n']} for k, v in e1['tabelle'].items()}).T
display(tab)
for quelle, liste in e1['groesster_abfall'].items():
    print(f'Größter Abfall, trainiert auf {quelle}:')
    display(pd.DataFrame(liste))

## 10b. Experiment E4: Lernkurve (beantwortet die Forschungsfrage)

Trainiert A je Stufe neu mit höchstens 0 / 25 / 50 / 100 / 150 / allen echten Trainingsausschnitten je Zusatzzeichen (8 Epochen je Stufe, zusammen etwa 2–2,5 Stunden auf der T4). Ergebnisse und Modelle liegen direkt in Drive (`dlacv_gtsrb/e4/n_*`); fertige Stufen werden übersprungen, nach einem Abbruch also einfach Abschnitte 1–4 und diese Zelle erneut ausführen.

In [ ]:
os.makedirs(f'{DRIVE_DIR}/e4', exist_ok=True)
!python scripts/auswertung/experiment_e4.py --config configs/colab_multitask.yaml --ziel "{DRIVE_DIR}/e4" 2>&1 | tee -a "{DRIVE_DIR}/e4/e4_log.txt"

In [ ]:
import json, pandas as pd
from IPython.display import Image as IPImage
with open(f'{DRIVE_DIR}/e4/lernkurve.json', encoding='utf-8') as f:
    kurve = json.load(f)
display(pd.DataFrame([{'Stufe': z['stufe'], 'echte Bilder': z['n_echt_train'],
                       **{k: z['echt_test'].get(k) for k in ('zusatz_exakt', 'zielgruppe_korrekt_relevant', 'zusatz_macro_f1', 'haupt_acc')}}
                      for z in kurve]))
display(IPImage(f'{DRIVE_DIR}/e4/lernkurve.png'))

## 10c. Gegenmaßnahme Risiko 1: „keins“ begrenzen

Mehr als die Hälfte der echten Trainingsausschnitte hat kein Zusatzzeichen. A wird mit höchstens 300 davon neu trainiert (15 Epochen wie A) und mit A verglichen. Etwa 1 Stunde auf der T4.

In [ ]:
import os
os.makedirs(f'{DRIVE_DIR}/keins300', exist_ok=True)
!python scripts/training/train_multitask.py --config configs/colab_multitask.yaml --max-keins 300 --ausgabe "{DRIVE_DIR}/keins300" 2>&1 | tee "{DRIVE_DIR}/keins300/log.txt"


In [ ]:
import json, pandas as pd
keys = ['zusatz_exakt', 'zielgruppe_korrekt_relevant', 'zusatz_macro_f1', 'haupt_acc', 'da_acc']
zeilen = {}
for name, pfad in [('A (alle)', 'multitask'), ('A, keins <= 300', 'keins300')]:
    with open(f'{DRIVE_DIR}/{pfad}/ergebnis.json', encoding='utf-8') as f:
        test = json.load(f)['echt_test']
    zeilen[name] = {k: test.get(k) for k in keys}
pd.DataFrame(zeilen).T.round(3)

## 11. Ergebnisse herunterladen

Das Modell danach lokal nach `models/checkpoints/` legen.

In [ ]:
!cd "{DRIVE_DIR}" && zip -q -r /content/multitask_ergebnisse.zip multitask zweistufig e1 e4 keins300 -x 'e4/*/modell.pth' 'keins300/modell.pth'
from google.colab import files
files.download('/content/multitask_ergebnisse.zip')